# RAMP Time Series Generation — Norte Amazonia Bolivia — 2050 projection

Copied unmodified from `analyse data ramp/reality/time_series.ipynb`, except `OUTPUT_DIR` is
suffixed to `output_energyscope_2050` (correction 4: no projection output can land in a 2025 path).
Executed for this horizon — unlike `demande.ipynb`/`home_systems.ipynb`, this notebook has no
horizon-specific input: it reuses the same (unchanged) `data ramp/` RAMP shape and Renewables.ninja
capacity-factor profiles as 2025, so its output is expected to be numerically identical to the
reality/2025 `Time_series.csv`.

This notebook builds the hourly time-series input files (`Time_series.csv`) required by EnergyScope for each of the 5 clusters, using the **reality** RAMP run.

For each cluster, RAMP minute-level load data is averaged to 8760 hourly values, summed across municipalities, and normalized to a fractional profile (sum = 1). Renewable capacity factors (PV, wind, solar) come from the same Renewables.ninja outputs as the sufficiency scenario. Mobility profiles are also identical.

Output: `output_energyscope_2050/C{k}/Time_series.csv`

## 1. Setup — paths, clusters, and column groups

`ELECTRICITY_COLS` lists all RAMP columns present in the reality run — **no** `big_school_*`, `health_center_*`, or `public_lighting_*` columns exist here.

`SPACE_COOLING_COLS` is reduced to `sufficiency_thermal_comfort` only (no school or health-center AC in the reality run).

In [1]:
import os
import pandas as pd
import numpy as np

RAMP_DIR       = "data ramp"
RENEWABLES_DIR = "../../renewable ninja/solar and wind/output"
OUTPUT_DIR     = "output_energyscope_2050"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir", "Puerto_Rico",
        "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando", "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# All electrical end-uses present in the reality run
# (big_school_*, health_center_*, public_lighting_* do NOT exist in this run)
ELECTRICITY_COLS = [
    "sufficiency_illumination", "sufficiency_ICT", "sufficiency_cold_storage",
    "sufficiency_thermal_comfort",
    "small_school_illumination", "small_school_ICT",
    "entertainment_business_illumination", "entertainment_business_ICT",
    "entertainment_business_cold_storage",
    "rice_processing_rice_processing",
    "restaurant_illumination", "restaurant_cold_storage", "restaurant_kitchen",
    "store_illumination", "store_ICT", "store_cold_storage",
    "workshop_illumination", "workshop_ICT", "workshop_machinery",
]

# Space cooling: only household thermal comfort (no school/health-center AC in reality run)
SPACE_COOLING_COLS = ["sufficiency_thermal_comfort"]

HOURS = pd.date_range("2019-01-01", periods=8760, freq="h", tz="UTC")

## 2. Helper functions

- **`load_ramp_hourly(muni, cols)`** — reads the reality CSV (1-minute steps, 525 600 rows), keeps only requested columns, and averages every 60 rows into one hourly value → 8760 rows.
- **`cluster_profile(munis, cols)`** — sums raw watt values across all municipalities in the cluster, then normalizes so the result sums to 1 (fractional time profile required by EnergyScope).

> The reality filename is `load_curve_energy_service_full_year_Norte_Amazonia_reality.csv` (different from the sufficiency filename).

In [2]:
def load_ramp_hourly(muni, cols):
    """Read a municipality's reality RAMP CSV and return 8760-row hourly averages."""
    path = f"{RAMP_DIR}/{muni}/load_curve_energy_service_full_year_Norte_Amazonia_reality.csv"
    if not os.path.exists(path):
        print(f"  Warning: file not found for {muni}")
        return None

    available = pd.read_csv(path, nrows=0).columns.tolist()
    valid_cols = [c for c in cols if c in available]
    if not valid_cols:
        return None

    df = pd.read_csv(path, usecols=valid_cols).fillna(0)
    # Average each 60-minute block into one hourly value
    hourly = pd.DataFrame({c: df[c].values.reshape(8760, 60).mean(axis=1) for c in valid_cols})
    del df
    return hourly


def cluster_profile(munis, cols):
    """Sum raw watts across all municipalities, then normalize to a unit profile (sum = 1)."""
    total = np.zeros(8760)
    for muni in munis:
        hourly = load_ramp_hourly(muni, cols)
        if hourly is not None:
            total += hourly.sum(axis=1).values
            del hourly

    if total.sum() == 0:
        print("  Warning: zero profile — using flat 1/8760")
        return np.full(8760, 1 / 8760)

    return total / total.sum()

## 3. Build the time series for each cluster

For each cluster:
1. Build the **electricity** and **space-cooling** profiles from reality RAMP data
2. Read renewable capacity factors (PV, wind, solar) — same files as the sufficiency scenario
3. Add mobility profiles (shared, loaded once)
4. Save to `output_energyscope_2050/C{k}/Time_series.csv`

> `WIND_OFFSHORE` copies `WIND_ONSHORE` — no offshore resource in this landlocked region, but the column is required by EnergyScope.  
> `HYDRO_DAM`, `HYDRO_RIVER`, `TIDAL`, `CSP` are set to 0.0001 (EnergyScope format placeholder).

In [3]:
# Mobility profiles — same for all clusters and identical to sufficiency scenario
mob = pd.read_csv("../data/Time_series.csv", sep=";", index_col=0)
mob_passenger = mob["MOBILITY_PASSENGER"].values / mob["MOBILITY_PASSENGER"].sum()
mob_freight   = mob["MOBILITY_FREIGHT"].values   / mob["MOBILITY_FREIGHT"].sum()

print("--- BUILDING TIME SERIES (reality) ---")

for k, munis in CLUSTERS.items():
    print(f"\nCluster {k}: {munis}")
    elec = cluster_profile(munis, ELECTRICITY_COLS)
    sc   = cluster_profile(munis, SPACE_COOLING_COLS)
    ren  = pd.read_csv(f"{RENEWABLES_DIR}/renewables_C{k}.csv")

    ts = pd.DataFrame({
        "ELECTRICITY":        elec,
        "SPACE_COOLING":      sc,
        "MOBILITY_PASSENGER": mob_passenger,
        "MOBILITY_FREIGHT":   mob_freight,
        "PV":                 ren["PV"].values,
        "WIND_ONSHORE":       ren["WIND_ONSHORE"].values,
        "WIND_OFFSHORE":      ren["WIND_ONSHORE"].values,  # no offshore in this region
        "HYDRO_DAM":          0.0001,
        "HYDRO_RIVER":        0.0001,
        "TIDAL":              0.0001,
        "SOLAR":              ren["SOLAR"].values,
        "CSP":                0.0001,
    }, index=HOURS)
    ts.index.name = ""

    out_path = f"{OUTPUT_DIR}/C{k}/Time_series.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    ts.to_csv(out_path, sep=";", date_format="%Y-%m-%d %H:%M:%S+00:00")
    print(f"  → saved to {out_path}")

print("\nDone.")

--- BUILDING TIME SERIES (reality) ---

Cluster 1: ['Exaltación', 'Reyes', 'Santa_Rosa_Beni', 'Ixiamas']


  → saved to output_energyscope_2050/C1/Time_series.csv

Cluster 2: ['Bolpebra']


  → saved to output_energyscope_2050/C2/Time_series.csv

Cluster 3: ['Guayaramerín', 'Riberalta', 'Puerto_Gonzalo_Moreno']


  → saved to output_energyscope_2050/C3/Time_series.csv

Cluster 4: ['Bella_Flor', 'Filadelfia', 'Ingavi', 'Nueva_Esperanza', 'Porvenir', 'Puerto_Rico', 'San_Lorenzo', 'San_Pedro', 'Santa_Rosa_Pando', 'Santos_Mercado', 'Sena', 'Villa_Nueva']


  → saved to output_energyscope_2050/C4/Time_series.csv

Cluster 5: ['Cobija']


  → saved to output_energyscope_2050/C5/Time_series.csv

Done.


## 4. Verification

The normalized profiles `ELECTRICITY`, `SPACE_COOLING`, `MOBILITY_PASSENGER`, `MOBILITY_FREIGHT` must each sum to **1.0**.  
Renewable columns (PV, WIND, SOLAR) will sum to their total annual capacity-factor hours — not 1.

In [4]:
print("--- VERIFICATION (normalized profiles should sum to 1.0) ---\n")
for k in CLUSTERS:
    ts = pd.read_csv(f"{OUTPUT_DIR}/C{k}/Time_series.csv", sep=";", index_col=0)
    sums = ts[["ELECTRICITY", "SPACE_COOLING", "MOBILITY_PASSENGER", "MOBILITY_FREIGHT"]].sum().round(6)
    print(f"C{k}: {sums.to_dict()}")

--- VERIFICATION (normalized profiles should sum to 1.0) ---

C1: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0}
C2: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0}
C3: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0}
C4: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0}
C5: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0}
